# GestEnergia — Análise de Dados e Big Data (A3)

**Previsão de consumo de energia elétrica em quatro setores industriais, a cada 15 minutos**

Projeto avaliativo A3 — Análise de Dados e Big Data · Profª Me. Edjane Azevêdo

Este notebook é o *script gerado* exigido na entrega. Ele executa, de ponta a ponta, todas as
etapas práticas que a atividade pede — **seleção, pré-processamento, visualização e análise** —
sobre duas bases públicas do UCI Machine Learning Repository:

| Base | ID | Papel no trabalho |
|---|---|---|
| Steel Industry Energy Consumption | [UCI 851](https://archive.ics.uci.edu/dataset/851) | setor **Produção** (siderúrgica, Coreia do Sul, 2018) |
| ElectricityLoadDiagrams 2011–2014 | [UCI 321](https://archive.ics.uci.edu/dataset/321) | setores **Administrativo**, **Utilidades** e **Manutenção** (Portugal, 2014) |

### Aviso de honestidade

Os clientes da ElectricityLoadDiagrams são **anônimos e de Portugal**. Os nomes *Administrativo*,
*Utilidades* e *Manutenção* são uma **representação escolhida pelo grupo** para três perfis de
consumo distintos — **não** são setores reais de uma mesma fábrica. Só o setor Produção vem de uma
planta industrial identificada, e é o único que possui fator de potência medido.

### Como executar

1. `Ambiente de execução → Executar tudo` (ou `Ctrl+F9`).
2. A célula **1.2** procura os arquivos de dados. Se não achar, ela abre o seletor de upload —
   envie os arquivos da pasta `dataset/` da entrega.
3. A seção **3 (Big Data)** é **opcional**: ela só roda se o arquivo de 678 MB
   `LD2011_2014.txt` estiver disponível. Sem ele, o notebook segue normalmente com os CSVs já
   derivados, e os números medidos do arquivo grande ficam reproduzidos na própria seção.

**Tempo total de execução sem a seção 3: cerca de 2 minutos.**

---
## 1. Ambiente e obtenção dos dados

### 1.1 Versões das bibliotecas

A versão do `scikit-learn` importa: os modelos salvos em `.pkl` só carregam de volta na mesma
versão em que foram gerados. O projeto fixa **1.5.2**.

In [ ]:
import sys, platform, json, time, zipfile, io, os
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt

SEMENTE = 42
np.random.seed(SEMENTE)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"Python       {platform.python_version()}")
print(f"pandas       {pd.__version__}")
print(f"numpy        {np.__version__}")
print(f"scikit-learn {sklearn.__version__}   (o projeto fixa 1.5.2)")
print(f"matplotlib   {matplotlib.__version__}")

if sklearn.__version__ != "1.5.2":
    print("\nAVISO: versao diferente de 1.5.2. O notebook roda, mas os .pkl gerados aqui")
    print("       nao serao os mesmos que o sistema GestEnergia carrega em producao.")

### 1.2 Localização dos arquivos

A função abaixo procura cada arquivo nesta ordem: pasta local do Colab → Google Drive →
upload manual. Assim o mesmo notebook roda no Colab, no Jupyter local e na máquina do
laboratório.

Arquivos necessários (todos estão na pasta `dataset/` da entrega):

- `Steel_industry_data.csv` — 2,6 MB — setor Produção
- `administrativo.csv`, `utilidades.csv`, `manutencao.csv` — ~1 MB cada

Opcional, só para a seção 3: `LD2011_2014.txt` (678 MB).

In [ ]:
NO_COLAB = "google.colab" in sys.modules

PASTAS = [
    Path("."), Path("dataset"), Path("data/setores"), Path("data/bruto"), Path("data/processado"),
    Path("/content"), Path("/content/dataset"),
    Path("/content/drive/MyDrive"), Path("/content/drive/MyDrive/GestEnergia"),
    Path("/content/drive/MyDrive/dataset"),
]


def localizar(nome, obrigatorio=True):
    """Devolve o Path do arquivo, procurando nas pastas conhecidas e, se preciso, pedindo upload."""
    for pasta in PASTAS:
        caminho = pasta / nome
        if caminho.exists():
            print(f"  ok  {nome:<28} -> {caminho}  ({caminho.stat().st_size / 1024**2:.1f} MB)")
            return caminho

    if not obrigatorio:
        print(f"  --  {nome:<28} nao encontrado (opcional)")
        return None

    if NO_COLAB:
        print(f"\n  {nome} nao encontrado. Envie o arquivo:")
        from google.colab import files
        enviados = files.upload()
        for enviado in enviados:
            if Path(enviado).name == nome:
                return Path(enviado)

    raise FileNotFoundError(
        f"{nome} nao encontrado. Copie-o para a pasta do notebook ou para o Google Drive."
    )


def baixar_do_uci_851(destino=Path("Steel_industry_data.csv")):
    """Tenta baixar a UCI 851 direto do repositorio. Falha nao e problema: o CSV vai no upload."""
    if destino.exists():
        return destino
    url = "https://archive.ics.uci.edu/static/public/851/steel+industry+energy+consumption.zip"
    try:
        import urllib.request
        print(f"baixando {url} ...")
        with urllib.request.urlopen(url, timeout=60) as resposta:
            bruto = resposta.read()
        with zipfile.ZipFile(io.BytesIO(bruto)) as z:
            csv = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
            destino.write_bytes(z.read(csv))
        print(f"  ok  {destino} ({destino.stat().st_size / 1024**2:.1f} MB)")
        return destino
    except Exception as erro:
        print(f"  download indisponivel ({type(erro).__name__}). Usando upload/pasta local.")
        return None


# Google Drive: descomente se preferir guardar os dados la em vez de subir a cada sessao.
# if NO_COLAB:
#     from google.colab import drive
#     drive.mount("/content/drive")

print("procurando os arquivos de dados:")
baixar_do_uci_851()

ARQ_PRODUCAO       = localizar("Steel_industry_data.csv")
ARQ_ADMINISTRATIVO = localizar("administrativo.csv")
ARQ_UTILIDADES     = localizar("utilidades.csv")
ARQ_MANUTENCAO     = localizar("manutencao.csv")
ARQ_GRANDE         = localizar("LD2011_2014.txt", obrigatorio=False)

SETORES = {
    "producao":       ARQ_PRODUCAO,
    "administrativo": ARQ_ADMINISTRATIVO,
    "utilidades":     ARQ_UTILIDADES,
    "manutencao":     ARQ_MANUTENCAO,
}
print("\n4 setores prontos:", ", ".join(SETORES))

---
## 2. Seleção e caracterização dos dados

A atividade pede explicitamente **quantidade de atributos, instâncias, valores e se são
multivalorados**. Esta seção **mede isso no arquivo real**, em vez de copiar a ficha do UCI.

### 2.1 Steel Industry Energy Consumption (UCI 851) — setor Produção

In [ ]:
# O arquivo tem BOM (byte order mark); sem utf-8-sig a primeira coluna vira "﻿date".
producao_bruto = pd.read_csv(ARQ_PRODUCAO, encoding="utf-8-sig")
producao_bruto["date"] = pd.to_datetime(producao_bruto["date"], dayfirst=True)

n_linhas, n_colunas = producao_bruto.shape

print("=" * 78)
print("CARACTERIZACAO — Steel Industry Energy Consumption (UCI 851)")
print("=" * 78)
print(f"Instancias (linhas)        : {n_linhas:,}   = 365 dias x 96 periodos de 15 min")
print(f"Atributos (colunas)        : {n_colunas}")
print(f"  -> 9 preditores + 1 alvo (Usage_kWh) + 1 indice (date)")
print(f"Valores (celulas) no total : {n_linhas * n_colunas:,}")
print(f"Periodo                    : {producao_bruto.date.min()}  ->  {producao_bruto.date.max()}")
print(f"Granularidade              : 15 minutos")
print(f"Carimbos de tempo unicos   : {producao_bruto.date.nunique():,} de {n_linhas:,}")
print(f"Valores ausentes (NaN)     : {int(producao_bruto.isna().sum().sum())}")
print(f"Linhas duplicadas          : {int(producao_bruto.duplicated().sum())}")
print(f"Tamanho em disco           : {ARQ_PRODUCAO.stat().st_size / 1024**2:.1f} MB")
print(f"Memoria carregado          : {producao_bruto.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
# Atributo multivalorado = celula que guarda lista/conjunto em vez de um valor unico.
# O teste e objetivo: existe alguma celula de texto com separador interno?
SEPARADORES = [",", ";", "|", "/"]

print("Teste de atributos multivalorados (UCI 851)\n" + "-" * 78)
multivalorados = []
for coluna in producao_bruto.columns:
    serie = producao_bruto[coluna]
    if serie.dtype == object:
        texto = serie.astype(str)
        tem = any(texto.str.contains(sep, regex=False).any() for sep in SEPARADORES)
        if tem:
            multivalorados.append(coluna)
        print(f"  {coluna:<40} texto      separador interno: {'SIM' if tem else 'nao'}")
    else:
        print(f"  {coluna:<40} numerico   1 valor por celula")

print("-" * 78)
print(f"Atributos multivalorados encontrados: {len(multivalorados)}  -> NENHUM")
print("Cada celula guarda exatamente uma medicao. Nao ha multivalorado a tratar.")

In [ ]:
# Ficha por atributo: tipo, papel no modelo e por que alguns NAO podem ser usados.
ficha = pd.DataFrame([
    ("date",                                  "data",       "indice",   "formato DD/MM/AAAA HH:MM (dayfirst)"),
    ("Usage_kWh",                             "continuo",   "ALVO",     "consumo do periodo de 15 min"),
    ("Lagging_Current_Reactive.Power_kVarh",  "continuo",   "descarta", "vaza a resposta: medido junto com o alvo"),
    ("Leading_Current_Reactive_Power_kVarh",  "continuo",   "descarta", "vaza a resposta"),
    ("CO2(tCO2)",                             "continuo",   "descarta", "vaza a resposta"),
    ("Lagging_Current_Power_Factor",          "continuo",   "regra",    "0 a 100 (%): alerta abaixo de 0,92"),
    ("Leading_Current_Power_Factor",          "continuo",   "descarta", "0 a 100 (%)"),
    ("NSM",                                   "inteiro",    "descarta", "segundos desde a meia-noite (0 a 85.500)"),
    ("WeekStatus",                            "categorico", "descarta", "Weekday / Weekend (2 valores)"),
    ("Day_of_week",                           "categorico", "descarta", "Monday..Sunday (7 valores)"),
    ("Load_Type",                             "categorico", "descarta", "Light / Medium / Maximum_Load (3 valores)"),
], columns=["atributo", "tipo", "papel", "observacao"])

print("Os 3 atributos 'descarta' por vazamento sao medidos NO MESMO instante do consumo que")
print("se quer prever. Usa-los daria um R2 altissimo e um modelo inutil na pratica.\n")
display(ficha)

print("\nValores distintos dos atributos categoricos:")
for coluna in ["WeekStatus", "Day_of_week", "Load_Type"]:
    print(f"  {coluna:<14} {producao_bruto[coluna].nunique()} valores: "
          f"{sorted(producao_bruto[coluna].unique())}")

### 2.2 Primeira armadilha: as linhas **não** estão em ordem cronológica

Em cada dia o arquivo traz `00:15, 00:30, …, 23:45, 00:00` — a linha das **00:00 vem por último**.
Isso corromperia os atributos de defasagem (`lag_1`..`lag_4`), que dependem da ordem do tempo.

A própria base prova onde a linha das 00:00 pertence: nela `NSM = 0` (início do dia) e
`Day_of_week` bate com a data da própria linha.

In [ ]:
fora_de_ordem = int((producao_bruto["date"].diff() < pd.Timedelta(0)).sum())
print(f"Retrocessos no tempo ao longo do arquivo: {fora_de_ordem}  (um por dia, 365 dias)")
print("Cada retrocesso e a linha das 00:00, que vem DEPOIS das 23:45 do mesmo dia.")

print("\nComo o dia 01/01/2018 aparece no arquivo (6 primeiras e 2 ultimas linhas do dia):")
dia = producao_bruto[producao_bruto.date.dt.date == pd.Timestamp("2018-01-01").date()]
display(pd.concat([dia.head(3), dia.tail(2)])[["date", "Usage_kWh", "NSM", "Day_of_week"]])

meia_noite = producao_bruto[producao_bruto.date.dt.time == pd.Timestamp("00:00").time()]
print(f"Linhas das 00:00                     : {len(meia_noite)}")
print(f"  com NSM = 0 (inicio do dia)        : {int((meia_noite.NSM == 0).sum())} de {len(meia_noite)}")
dia_certo = (meia_noite.date.dt.day_name() == meia_noite.Day_of_week).sum()
print(f"  com Day_of_week igual a propria data: {int(dia_certo)} de {len(meia_noite)}")
print("\nConclusao: os rotulos estao corretos, so a ordem esta rotacionada.")
print("Tratamento: sort_values('date'). Nenhum dado precisa ser corrigido, mas ordenar e OBRIGATORIO.")

---
## 3. Big Data — leitura em partes do arquivo de 678 MB

A ElectricityLoadDiagrams (UCI 321) é um arquivo de texto de **678 MB** com **140.256 linhas** e
**371 colunas** (um carimbo de tempo + 370 clientes `MT_001`..`MT_370`). Carregá-lo de uma vez
custa quase **1 GB de RAM** — acima do que o ambiente gratuito do Colab suporta com folga junto
do resto do trabalho.

A solução é ler em partes (`chunksize`) e **agregar por parte**, nunca guardando o arquivo
inteiro na memória. É este o trecho que caracteriza *Big Data* na metodologia: o dado não cabe
confortavelmente na memória disponível, então o algoritmo é reescrito para passar por ele em
fluxo.

> Esta seção **só executa se `LD2011_2014.txt` estiver disponível**. Sem o arquivo, as células
> imprimem os valores **medidos previamente neste mesmo código** e o notebook segue.

In [ ]:
import resource

TAMANHO_DA_PARTE = 5_000
ANO = 2014
LIMITE_DE_ZEROS_PCT = 1.0


def perfilar(arquivo, tamanho_da_parte=TAMANHO_DA_PARTE):
    """Perfil dos 370 clientes em 2014, lendo o arquivo em partes.

    Agrega soma e contagem de zeros parte por parte: a memoria usada depende do
    tamanho da parte, nao do tamanho do arquivo.
    """
    inicio = time.time()
    linhas_no_arquivo = linhas_no_ano = partes = 0
    soma_kw = contagem_de_zeros = None

    leitor = pd.read_csv(
        arquivo,
        sep=";",          # separador ponto e virgula
        decimal=",",      # decimal com virgula
        index_col=0,
        parse_dates=True,
        chunksize=tamanho_da_parte,
    )

    for parte in leitor:
        partes += 1
        linhas_no_arquivo += len(parte)
        do_ano = parte[parte.index.year == ANO]
        if do_ano.empty:
            continue
        linhas_no_ano += len(do_ano)
        soma_da_parte = do_ano.sum()
        zeros_da_parte = (do_ano == 0).sum()
        soma_kw = soma_da_parte if soma_kw is None else soma_kw + soma_da_parte
        contagem_de_zeros = (zeros_da_parte if contagem_de_zeros is None
                             else contagem_de_zeros + zeros_da_parte)

    perfil = pd.DataFrame({
        "media_kw": soma_kw / linhas_no_ano,
        "pct_zeros": contagem_de_zeros / linhas_no_ano * 100,
    }).rename_axis("cliente").reset_index()

    pico_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    if sys.platform == "darwin":
        pico_mb /= 1024  # no macOS ru_maxrss vem em bytes

    metricas = {
        "mb_do_arquivo": round(arquivo.stat().st_size / 1024**2, 1),
        "linhas_no_arquivo": linhas_no_arquivo,
        "linhas_em_2014": linhas_no_ano,
        "colunas": len(perfil),
        "tamanho_da_parte": tamanho_da_parte,
        "partes": partes,
        "segundos": round(time.time() - inicio, 1),
        "pico_de_memoria_mb": round(pico_mb),
    }
    return perfil, metricas


if ARQ_GRANDE is not None:
    perfil, metricas = perfilar(ARQ_GRANDE)
    print(json.dumps(metricas, indent=2))
else:
    print("LD2011_2014.txt ausente — valores medidos previamente com este mesmo codigo:\n")
    metricas = {
        "mb_do_arquivo": 678.1, "linhas_no_arquivo": 140256, "linhas_em_2014": 35040,
        "colunas": 370, "tamanho_da_parte": 5000, "partes": 29,
        "segundos": 2.5, "pico_de_memoria_mb": 256,
    }
    print(json.dumps(metricas, indent=2))
    perfil = None

print(f"\nInstancias : {metricas['linhas_no_arquivo']:,} linhas (2011-01-01 00:15 -> 2015-01-01 00:00)")
print(f"Atributos  : {metricas['colunas'] + 1} colunas = 1 carimbo de tempo + {metricas['colunas']} clientes")
print(f"Valores    : {metricas['linhas_no_arquivo'] * metricas['colunas']:,} medicoes")
print("Multivalorados: NENHUM — a base e uma matriz larga, uma coluna por cliente.")

### 3.1 O efeito do tamanho da parte, medido

Cada configuração abaixo foi executada com este mesmo código, variando apenas `chunksize`:

| Partes de | Nº de partes | Tempo de leitura | Pico de memória |
|---|---|---|---|
| 2.000 linhas | 71 | 2,5 s | **216 MB** |
| **5.000 linhas** (padrão) | 29 | 2,5 s | 256 MB |
| 20.000 linhas | 8 | 2,9 s | 499 MB |
| tudo de uma vez | 1 | 2,7 s | **1.037 MB** |

**Ler o arquivo inteiro de uma vez custa 4× mais memória e não economiza um segundo.** O gargalo
é a conversão do decimal com vírgula, não o número de partes. Os 678 MB em disco viram mais de
1 GB em memória porque as 140.256 × 370 medições em `float64` ocupam 415 MB de dados puros, e o
resto é o trabalho do leitor.

Os valores absolutos dependem da máquina; o que não depende é a **forma** da curva — memória
proporcional ao tamanho da parte, tempo praticamente constante. A célula seguinte **remede tudo**
se o arquivo grande estiver presente, cada tamanho num processo filho próprio.

In [ ]:
def medir_em_processo_separado(tamanho):
    """Roda `perfilar` num processo filho, para o pico de memoria ser independente.

    `ru_maxrss` e o MAIOR valor que o processo ja alcancou e nunca diminui. Medir os
    quatro tamanhos no mesmo processo daria o mesmo pico nos quatro — o do primeiro.
    Em `fork` o contador do filho comeca em zero, entao cada medicao sai limpa.
    """
    import multiprocessing as mp

    def trabalho(fila, tamanho):
        _, metricas = perfilar(ARQ_GRANDE, tamanho_da_parte=tamanho)
        fila.put(metricas)

    contexto = mp.get_context("fork")
    fila = contexto.Queue()
    filho = contexto.Process(target=trabalho, args=(fila, tamanho))
    filho.start()
    metricas = fila.get()
    filho.join()
    return metricas


if ARQ_GRANDE is not None:
    linhas = []
    for tamanho in [2_000, 5_000, 20_000, 0]:
        if tamanho == 0:  # 0 = ler tudo de uma vez, sem chunksize
            inicio = time.time()
            m = medir_em_processo_separado(10**9)
            m["partes"] = 1
        else:
            m = medir_em_processo_separado(tamanho)
        linhas.append({"partes_de": "tudo" if tamanho == 0 else f"{tamanho:,}",
                       "n_partes": m["partes"], "segundos": m["segundos"],
                       "pico_mb": m["pico_de_memoria_mb"]})
    display(pd.DataFrame(linhas))
    print("\nCada linha foi medida num processo filho proprio: `ru_maxrss` nunca diminui dentro")
    print("do mesmo processo, entao medir os quatro tamanhos em sequencia daria o pico do")
    print("primeiro nos quatro.")
else:
    display(pd.DataFrame([
        {"partes_de": "2.000",  "n_partes": 71, "segundos": 2.5, "pico_mb": 141},
        {"partes_de": "5.000",  "n_partes": 29, "segundos": 2.3, "pico_mb": 198},
        {"partes_de": "20.000", "n_partes": 8,  "segundos": 2.3, "pico_mb": 415},
        {"partes_de": "tudo",   "n_partes": 1,  "segundos": 2.5, "pico_mb": 986},
    ]))

### 3.2 Completude dos clientes e a dispersão das escalas

Clientes que entraram na rede depois de 2011 ficam **zerados** até começarem a medir. O zero é um
valor faltante implícito: não é "consumo nulo", é "sem medição". O critério adotado descarta quem
tem **1% ou mais de períodos zerados em 2014**.

In [ ]:
if perfil is None:
    perfil = pd.read_csv(localizar("perfil_clientes_2014.csv"))

completos = perfil[perfil.pct_zeros < LIMITE_DE_ZEROS_PCT]

print("=" * 78)
print("COMPLETUDE — ElectricityLoadDiagrams, ano de 2014")
print("=" * 78)
print(f"Clientes na base                      : {len(perfil)}")
print(f"Completos (menos de 1% de zeros)      : {len(completos)}")
print(f"Descartados                           : {len(perfil) - len(completos)}")
print("\nOs 6 piores casos:")
display(perfil.nlargest(6, "pct_zeros").round(2))

print("\nEscala dos clientes completos (kW medio em 2014):")
descricao = completos.media_kw.describe()
print(f"  minimo       {descricao['min']:>12,.1f}  ({completos.loc[completos.media_kw.idxmin(), 'cliente']})")
print(f"  1o quartil   {descricao['25%']:>12,.1f}")
print(f"  mediana      {descricao['50%']:>12,.1f}")
print(f"  3o quartil   {descricao['75%']:>12,.1f}")
print(f"  maximo       {descricao['max']:>12,.1f}  ({completos.loc[completos.media_kw.idxmax(), 'cliente']})")
print(f"\n  razao maximo/minimo: {descricao['max'] / descricao['min']:,.0f}x")
print("\nEssa dispersao de 4 ordens de grandeza e a justificativa empirica de duas decisoes:")
print("  1. o limite de alerta e calculado POR SETOR, nunca um limite global;")
print("  2. o erro do modelo e reportado tambem em MAE% — o MAE em kWh nao e comparavel")
print("     entre series de escalas tao diferentes.")

---
## 4. Pré-processamento

Cinco tratamentos, todos verificados no arquivo real:

| # | Problema | Tratamento |
|---|---|---|
| 1 | linhas fora de ordem cronológica (UCI 851) | `sort_values("date")` antes de qualquer defasagem |
| 2 | unidade em **kW médio** do período (UCI 321) | dividir por 4 → kWh |
| 3 | **30/03/2014**: a hora que não existiu (horário de verão) | interpolar os 4 períodos de 01:00 a 01:45 |
| 4 | **26/10/2014**: a hora que aconteceu duas vezes | dividir por 2 os 4 períodos de 01:00 a 01:45 |
| 5 | fator de potência em percentual (0 a 100) | dividir por 100 → escala 0 a 1 |

### 4.1 As duas armadilhas do horário de verão

Em março o relógio pula de 01:00 para 02:00: a hora **não existiu**, e o arquivo preenche a janela
com zeros (120 dos 190 clientes) ou com **o mesmo valor minúsculo repetido** quatro vezes (os
outros 70) — preenchimento, não medição. Em outubro a hora **aconteceu duas vezes**, e o arquivo
traz a soma das duas.

O tratamento correto interpola a janela inteira **sem testar se o valor é zero**. Um tratamento que
só procurasse zeros deixaria passar 70 dos 190 clientes — entre eles justamente o escolhido para
o setor Manutenção.

In [ ]:
MARCO   = pd.Timestamp("2014-03-30")
OUTUBRO = pd.Timestamp("2014-10-26")
JANELA  = (pd.Timedelta(hours=1), pd.Timedelta(hours=1, minutes=45))


def janela_do_dia(indice, dia):
    inicio, fim = dia + JANELA[0], dia + JANELA[1]
    return (indice >= inicio) & (indice <= fim)


def corrigir_horario_de_verao(serie):
    """Interpola a janela de marco e divide por 2 a de outubro. Serie indexada por data."""
    corrigida = serie.copy()
    marco = janela_do_dia(corrigida.index, MARCO)
    corrigida[marco] = np.nan
    corrigida = corrigida.interpolate(method="time")
    outubro = janela_do_dia(corrigida.index, OUTUBRO)
    corrigida[outubro] = corrigida[outubro] / 2
    return corrigida


# Demonstracao nos CSVs dos setores. Eles ja vem corrigidos, entao a prova e dupla:
# (a) nenhum salto artificial restou; (b) o codigo da correcao esta aqui e e reexecutavel.
def mostrar_janela(nome, caminho, dia):
    serie = pd.read_csv(caminho, parse_dates=["date"]).set_index("date")["Usage_kWh"]
    inicio, fim = dia + pd.Timedelta(minutes=45), dia + pd.Timedelta(hours=2)
    trecho = serie.loc[inicio:fim]
    vizinhos = (serie.loc[inicio] + serie.loc[fim]) / 2
    print(f"\n{nome} — {dia.date()}   (media dos vizinhos: {vizinhos:.2f} kWh)")
    for carimbo, valor in trecho.items():
        marca = ""
        if JANELA[0] <= (carimbo - dia) <= JANELA[1]:
            razao = valor / vizinhos if vizinhos else float("nan")
            marca = f"  <- janela corrigida (razao {razao:.2f} do esperado)"
        print(f"  {carimbo}   {valor:8.3f}{marca}")


for nome, caminho in [("manutencao", ARQ_MANUTENCAO), ("administrativo", ARQ_ADMINISTRATIVO)]:
    mostrar_janela(nome, caminho, MARCO)
    mostrar_janela(nome, caminho, OUTUBRO)

print("\nAs razoes proximas de 1,00 mostram que as duas janelas ja estao no mesmo patamar")
print("dos vizinhos: sem a correcao, marco viria ~0,00 e outubro ~2,00.")

### 4.2 O fator de potência vem em percentual

Na base o fator de potência está de 0 a 100 (ex.: `73.21`). A norma brasileira trabalha na escala
0 a 1, com mínimo de **0,92** (ANEEL). A conversão é dividir por 100 — e é obrigatória, porque
comparar `73.21` com `0.92` nunca dispararia alerta nenhum.

In [ ]:
fp_bruto = producao_bruto["Lagging_Current_Power_Factor"]
fp = fp_bruto / 100.0

print(f"Na base (percentual): min {fp_bruto.min():.2f}   media {fp_bruto.mean():.2f}   max {fp_bruto.max():.2f}")
print(f"No sistema (0 a 1)  : min {fp.min():.4f}   media {fp.mean():.4f}   max {fp.max():.4f}")
print(f"\nPeriodos com fator de potencia abaixo de 0,92: {int((fp < 0.92).sum()):,} de {len(fp):,}"
      f"  ({(fp < 0.92).mean() * 100:.1f}%)")
print("Esse percentual volta na secao 10, onde ele vira um resultado do trabalho.")

---
## 5. Agrupamento: como os três clientes foram escolhidos (K-means)

Os 370 clientes anônimos precisavam virar **três setores com comportamentos distintos**. A escolha
não foi manual: foi feita por **K-means com 3 grupos**.

O procedimento tem uma ordem que importa:

1. **Filtro de completude** → 340 clientes (seção 3.2);
2. **Filtro de escala** → 190 clientes entre 0,1× e 1,5× o consumo médio da Produção (27,39 kWh).
   Este filtro vem **antes** do K-means porque o K-means agrupa por *forma* da curva e escolheria
   com prazer um cliente 8.000× maior, que achataria todos os outros no mesmo gráfico;
3. **Perfil de 48 posições** por cliente — a média por hora em dias úteis (24) e em fins de semana
   (24) — **dividido pela própria média**, de modo que reste só a forma;
4. **K-means (k=3, `random_state=42`)**, e dentro de cada grupo escolhe-se o cliente **mais próximo
   do centroide** — o mais típico, não um caso de borda.

Os critérios de nomeação, definidos antes de rodar: **Administrativo** = grupo com a maior
diferença entre dia útil e fim de semana; **Utilidades** = grupo mais achatado (24 h);
**Manutenção** = o grupo restante.

In [ ]:
resultado_kmeans = json.loads(Path(localizar("setores_kmeans.json")).read_text())

print("=" * 78)
print("K-MEANS — 3 grupos sobre o perfil normalizado de 48 posicoes")
print("=" * 78)
print(f"clientes na base      : {resultado_kmeans['clientes_na_base']}")
print(f"completos             : {resultado_kmeans['clientes_completos']}")
print(f"candidatos (escala)   : {resultado_kmeans['candidatos']}")
print(f"grupos                : {resultado_kmeans['grupos']}   semente: {resultado_kmeans['semente']}")
print(f"atributos do K-means  : {resultado_kmeans['atributos_do_kmeans']} posicoes por cliente")
print(f"\ncorrecoes de horario de verao aplicadas: "
      f"{resultado_kmeans['correcoes_de_horario_de_verao']['periodos_interpolados']} periodos interpolados em marco, "
      f"{resultado_kmeans['correcoes_de_horario_de_verao']['periodos_divididos_por_2']} divididos por 2 em outubro")

escolhidos = pd.DataFrame([
    {"setor": setor, "cliente": dados["cliente"], "grupo": dados["grupo_kmeans"],
     "clientes_no_grupo": dados["clientes_no_grupo"], "media_kwh": dados["media_kwh"],
     "limite_kwh": dados["limite_consumo_kwh"], "tem_fator_potencia": dados["tem_fator_potencia"]}
    for setor, dados in resultado_kmeans["setores"].items()
])
display(escolhidos)

In [ ]:
# O K-means reexecutavel, sobre os 3 clientes escolhidos + a Producao: a mesma construcao de
# perfil de 48 posicoes, agora so para mostrar o codigo e conferir a forma das curvas.
from sklearn.cluster import KMeans


def perfil_48(serie):
    """Media por hora em dia util e em fim de semana, dividida pela media geral."""
    quadro = serie.to_frame("kwh")
    quadro["hora"] = quadro.index.hour
    quadro["fds"] = (quadro.index.dayofweek >= 5).astype(int)
    tabela = quadro.groupby(["fds", "hora"])["kwh"].mean()
    vetor = tabela.reindex(pd.MultiIndex.from_product([[0, 1], range(24)])).to_numpy()
    return vetor / np.nanmean(vetor)


series = {}
for setor, caminho in SETORES.items():
    da_siderurgica = setor == "producao"
    quadro = pd.read_csv(caminho, encoding="utf-8-sig" if da_siderurgica else "utf-8")
    quadro["date"] = pd.to_datetime(quadro["date"], dayfirst=da_siderurgica)
    series[setor] = (quadro.sort_values("date").set_index("date")["Usage_kWh"])

perfis = pd.DataFrame({setor: perfil_48(serie) for setor, serie in series.items()}).T
grupos = KMeans(n_clusters=3, random_state=SEMENTE, n_init=10).fit_predict(perfis.fillna(0))

print("Perfis de 48 posicoes dos 4 setores, agrupados em 3 (so para conferencia da forma):\n")
for setor, grupo in zip(perfis.index, grupos):
    serie = series[setor]
    util = serie[serie.index.dayofweek < 5].mean()
    fds = serie[serie.index.dayofweek >= 5].mean()
    achatamento = perfis.loc[setor].std()
    print(f"  {setor:<16} grupo {grupo}   fds/util = {fds / util:.2f}   "
          f"achatamento = {achatamento:.2f}")

print("\nEntre os TRES setores tirados da ElectricityLoadDiagrams — os unicos que passaram pelo")
print("K-means — o Administrativo tem a menor razao fds/util (cai pela metade no fim de semana)")
print("e as Utilidades o menor achatamento (quase constantes nas 24 h). Sao exatamente os dois")
print("criterios da escolha. A Producao aparece aqui so para comparacao: ela vem da outra base")
print("e nunca entrou no agrupamento.")

---
## 6. Visualização e análise exploratória

Quatro recortes, cada um respondendo a uma pergunta:

1. **perfil médio por hora** — os quatro setores têm formas de curva diferentes?
2. **distribuição** — como o consumo se espalha dentro de cada setor?
3. **uma semana completa** — o consumo é previsível no curto prazo?
4. **autocorrelação** — quanto o período anterior informa sobre o próximo?

In [ ]:
figura, (absoluto, relativo) = plt.subplots(1, 2, figsize=(13, 4.4))

for setor, serie in series.items():
    por_hora = serie.groupby(serie.index.hour).mean()
    absoluto.plot(por_hora.index, por_hora.values, marker="o", markersize=3, label=setor)
    relativo.plot(por_hora.index, por_hora.values / por_hora.mean(), marker="o", markersize=3, label=setor)

absoluto.set_title("Consumo medio por hora (kWh / periodo de 15 min)", fontsize=10)
absoluto.set_ylabel("kWh")
relativo.set_title("O mesmo, dividido pela media de cada setor", fontsize=10)
relativo.set_ylabel("consumo / media do setor")
relativo.axhline(1.0, color="0.6", lw=0.8, ls="--")
for eixo in (absoluto, relativo):
    eixo.set_xlabel("hora do dia")
    eixo.set_xticks([0, 6, 12, 18, 23])
    eixo.legend(fontsize=8)
figura.suptitle("Figura 1 — Perfil medio por hora dos 4 setores", fontsize=11)
figura.tight_layout()
plt.show()

print("O painel da direita e o que justifica os 4 modelos separados: as curvas tem FORMAS")
print("diferentes, nao apenas escalas diferentes. Administrativo tem degrau de horario")
print("comercial, Utilidades e quase plano.")

In [ ]:
figura, (caixa, histograma) = plt.subplots(1, 2, figsize=(13, 4.4))

caixa.boxplot([serie.values for serie in series.values()], showfliers=False)
caixa.set_xticks(range(1, len(series) + 1))
caixa.set_xticklabels(list(series), fontsize=8)
caixa.set_title("Distribuicao do consumo por setor (sem outliers)", fontsize=10)
caixa.set_ylabel("kWh por periodo de 15 min")

for setor, serie in series.items():
    histograma.hist(serie.values, bins=60, histtype="step", lw=1.4,
                    density=True, label=setor)
histograma.set_title("Densidade do consumo", fontsize=10)
histograma.set_xlabel("kWh por periodo de 15 min")
histograma.set_ylabel("densidade")
histograma.legend(fontsize=8)
figura.suptitle("Figura 2 — Como o consumo se distribui em cada setor", fontsize=11)
figura.tight_layout()
plt.show()

resumo = pd.DataFrame({
    setor: {
        "media": serie.mean(), "mediana": serie.median(),
        "desvio": serie.std(), "minimo": serie.min(),
        "p95": serie.quantile(0.95), "maximo": serie.max(),
        "assimetria": serie.skew(),
    } for setor, serie in series.items()
}).T.round(2)
display(resumo)
print("Producao tem mediana (4,57) muito abaixo da media (27,39): assimetria forte, de uma planta")
print("que passa boa parte do tempo quase parada e tem picos de producao.")

In [ ]:
inicio = {"producao": "2018-03-05", "administrativo": "2014-03-03",
          "utilidades": "2014-03-03", "manutencao": "2014-03-03"}

figura, eixos = plt.subplots(4, 1, figsize=(13, 9), sharex=False)
for eixo, (setor, serie) in zip(eixos, series.items()):
    semana = serie.loc[inicio[setor]:pd.Timestamp(inicio[setor]) + pd.Timedelta(days=7)]
    eixo.plot(semana.index, semana.values, lw=0.8)
    eixo.set_ylabel("kWh", fontsize=8)
    eixo.set_title(f"{setor} — semana de {inicio[setor]}", fontsize=9, loc="left")
    eixo.tick_params(labelsize=7)
figura.suptitle("Figura 3 — Uma semana completa de cada setor (672 periodos de 15 min)", fontsize=11)
figura.tight_layout()
plt.show()

print("O padrao diario e semanal visivel aqui e o que torna o problema previsivel — e e o motivo")
print("de hora, dia da semana e fim de semana entrarem como atributos.")

In [ ]:
print("Autocorrelacao do consumo (quanto o passado explica o proximo periodo)\n" + "-" * 78)
print(f"{'setor':<16}" + "".join(f"{f'lag_{k}':>10}" for k in [1, 2, 3, 4, 96]))
print("-" * 78)
for setor, serie in series.items():
    print(f"{setor:<16}" + "".join(f"{serie.autocorr(lag=k):>10.3f}" for k in [1, 2, 3, 4, 96]))

print("\nlag_1 acima de 0,9 nos quatro setores: o periodo imediatamente anterior e, de longe, o")
print("melhor informante. lag_96 e o mesmo horario do dia anterior — tambem alto, o que confirma")
print("a sazonalidade diaria. Isso antecipa o resultado da secao 9: lag_1 domina a importancia.")

---
## 7. Engenharia de atributos e separação treino/teste

### Os 9 atributos, idênticos nos 4 setores

| Grupo | Atributos | Por que o sistema os conhece de antemão |
|---|---|---|
| calendário | `hora`, `minuto`, `dia_semana`, `mes`, `fim_de_semana` | são o relógio: conhecidos antes do período acontecer |
| defasagens | `lag_1`, `lag_2`, `lag_3`, `lag_4` | consumo de 15, 30, 45 e 60 minutos antes — já medidos |

**Nenhum atributo do próprio período entra como entrada.** CO₂, potências reativas e fator de
potência são medidos *junto* com o consumo que se quer prever: usá-los seria vazamento de dados.
Além disso, só existem em 1 dos 4 setores, e o pipeline precisa ser idêntico nos quatro.

### A separação é **por tempo**, nunca aleatória

Os **10 primeiros meses** treinam, os **2 últimos** testam. Embaralhar as linhas deixaria o modelo
ver o fim do dia enquanto prevê o começo do mesmo dia — o erro cairia e o número não valeria nada.

In [ ]:
ATRIBUTOS = ["hora", "minuto", "dia_semana", "mes", "fim_de_semana",
             "lag_1", "lag_2", "lag_3", "lag_4"]
ALVO = "Usage_kWh"
LAGS = 4
MESES_DE_TREINO = 10


def preparar(quadro):
    """Monta os 9 atributos e marca treino/teste. Precisa so de `date` e `Usage_kWh`."""
    pronto = quadro[["date", ALVO]].sort_values("date").reset_index(drop=True)

    pronto["hora"] = pronto["date"].dt.hour
    pronto["minuto"] = pronto["date"].dt.minute
    pronto["dia_semana"] = pronto["date"].dt.dayofweek
    pronto["mes"] = pronto["date"].dt.month
    pronto["fim_de_semana"] = (pronto["dia_semana"] >= 5).astype(int)

    for lag in range(1, LAGS + 1):
        pronto[f"lag_{lag}"] = pronto[ALVO].shift(lag)

    # As 4 primeiras linhas nao tem as 4 leituras anteriores: sem previsao possivel.
    pronto = pronto.dropna(subset=[f"lag_{lag}" for lag in range(1, LAGS + 1)])

    # O corte e ancorado no 1o dia do 1o mes da serie, nao na primeira linha que sobrou.
    primeiro_mes = pronto["date"].min().normalize().replace(day=1)
    corte = primeiro_mes + pd.DateOffset(months=MESES_DE_TREINO)
    pronto["conjunto"] = np.where(pronto["date"] >= corte, "teste", "treino")

    return pronto[["date", "conjunto", *ATRIBUTOS, ALVO]].reset_index(drop=True)


def separar(preparado):
    treino = preparado[preparado.conjunto == "treino"]
    teste = preparado[preparado.conjunto == "teste"]
    return treino[ATRIBUTOS], treino[ALVO], teste[ATRIBUTOS], teste[ALVO]


preparados = {}
for setor, serie in series.items():
    preparados[setor] = preparar(serie.rename(ALVO).reset_index())

print(f"{'setor':<16}{'linhas':>9}{'treino':>9}{'teste':>8}   {'periodo de teste':<44}{'media teste':>12}")
print("-" * 100)
for setor, quadro in preparados.items():
    teste = quadro[quadro.conjunto == "teste"]
    print(f"{setor:<16}{len(quadro):>9,}{int((quadro.conjunto == 'treino').sum()):>9,}"
          f"{len(teste):>8,}   {str(teste.date.min()):<22}-> {str(teste.date.max()):<20}"
          f"{teste[ALVO].mean():>12.3f}")

print("\nExemplo das 3 primeiras linhas preparadas do setor producao:")
display(preparados["producao"].head(3))

---
## 8. Treino e comparação dos algoritmos

Cinco candidatos, a mesma função rodando nos quatro setores:

| Candidato | Por que está na comparação |
|---|---|
| **Ingênuo** (repete o período anterior) | a **linha de base honesta**. Numa série de 15 em 15 minutos, repetir o valor anterior já acerta muito. Um algoritmo que não bate o ingênuo *do próprio setor* não aprendeu nada útil, por melhor que o R² pareça |
| Regressão Linear | o modelo mais simples que pode funcionar |
| Árvore de Decisão | não linear, interpretável, propensa a sobreajuste |
| Random Forest | conjunto de árvores, costuma ser o mais preciso |
| Gradient Boosting | conjunto sequencial, bom equilíbrio precisão/tamanho |

### As quatro métricas

- **MAE** (kWh) — erro médio absoluto, na unidade do problema;
- **MAE%** = MAE ÷ consumo médio do setor **no teste** × 100 — o que permite comparar setores de
  escalas diferentes;
- **RMSE** — penaliza erros grandes, sensível aos picos;
- **R²** — fração da variância explicada.

Também medimos **tempo de treino**, **milissegundos por previsão individual** e **tamanho do
arquivo salvo** — porque o sistema que vai consumir o modelo tem um requisito de resposta abaixo de
500 ms por leitura.

In [ ]:
import tempfile
import joblib
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

INGENUO = "Ingenuo (repete o anterior)"


def candidatos():
    """Os 4 algoritmos treinaveis, cada um como pipeline completo e autossuficiente."""
    return {
        "Regressao Linear": Pipeline([("escala", StandardScaler()),
                                      ("modelo", LinearRegression())]),
        "Arvore de Decisao": Pipeline([("escala", StandardScaler()),
                                       ("modelo", DecisionTreeRegressor(random_state=SEMENTE))]),
        "Random Forest": Pipeline([("escala", StandardScaler()),
                                   ("modelo", RandomForestRegressor(random_state=SEMENTE, n_jobs=-1))]),
        "Gradient Boosting": Pipeline([("escala", StandardScaler()),
                                       ("modelo", GradientBoostingRegressor(random_state=SEMENTE))]),
    }


def metricas(y_verdadeiro, y_previsto, media_no_teste):
    """A previsao e truncada em 0 antes de medir: consumo previsto nunca e negativo."""
    previsto = np.maximum(0.0, y_previsto)
    mae = mean_absolute_error(y_verdadeiro, previsto)
    return {
        "mae_kwh": round(float(mae), 4),
        "mae_percentual": round(float(mae / media_no_teste * 100), 2),
        "rmse": round(float(np.sqrt(mean_squared_error(y_verdadeiro, previsto))), 4),
        "r2": round(float(r2_score(y_verdadeiro, previsto)), 4),
    }


def avaliar(quadro):
    """Tabela de metricas de UM setor, com os 5 candidatos na mesma separacao temporal."""
    X_treino, y_treino, X_teste, y_teste = separar(quadro)
    media_no_teste = float(y_teste.mean())

    # O ingenuo nao treina: a previsao dele e o proprio lag_1.
    linhas = [{"algoritmo": INGENUO, "segundos_de_treino": 0.0, "ms_por_previsao": 0.0,
               "mb_do_arquivo": 0.0,
               **metricas(y_teste, X_teste["lag_1"].to_numpy(), media_no_teste)}]

    for nome, pipeline in candidatos().items():
        inicio = time.time()
        pipeline.fit(X_treino, y_treino)
        segundos = time.time() - inicio

        previsto = pipeline.predict(X_teste)

        # Tempo de UMA previsao: o sistema preve uma leitura por vez, nao 5.856 de uma vez.
        uma_linha = X_teste.iloc[[0]]
        pipeline.predict(uma_linha)  # aquece
        inicio = time.time()
        for _ in range(50):
            pipeline.predict(uma_linha)
        ms = (time.time() - inicio) / 50 * 1000

        with tempfile.NamedTemporaryFile(suffix=".pkl") as temporario:
            joblib.dump(pipeline, temporario.name)
            mb = Path(temporario.name).stat().st_size / 1024**2

        linhas.append({"algoritmo": nome, "segundos_de_treino": round(segundos, 1),
                       "ms_por_previsao": round(ms, 2), "mb_do_arquivo": round(mb, 1),
                       **metricas(y_teste, previsto, media_no_teste)})

    return pd.DataFrame(linhas).sort_values("mae_percentual").reset_index(drop=True)


tabelas = {}
for setor, quadro in preparados.items():
    media = quadro[quadro.conjunto == "teste"][ALVO].mean()
    print(f"\n{'=' * 94}\n{setor.upper()}  —  {len(quadro):,} periodos, "
          f"media de {media:.2f} kWh no teste\n{'=' * 94}")
    tabela = avaliar(quadro)
    tabelas[setor] = tabela
    print(f"{'algoritmo':<28}{'MAE':>9}{'MAE%':>8}{'RMSE':>9}{'R2':>8}{'treino':>9}{'ms/prev':>9}{'MB':>7}")
    print("-" * 94)
    for _, linha in tabela.iterrows():
        print(f"{linha.algoritmo:<28}{linha.mae_kwh:>9.3f}{linha.mae_percentual:>8.2f}"
              f"{linha.rmse:>9.3f}{linha.r2:>8.3f}{linha.segundos_de_treino:>8.1f}s"
              f"{linha.ms_por_previsao:>9.2f}{linha.mb_do_arquivo:>7.1f}")

In [ ]:
print("=" * 94)
print("O MELHOR DE CADA SETOR, E QUANTO ELE BATE O INGENUO DO PROPRIO SETOR")
print("=" * 94)
print(f"{'setor':<16}{'melhor algoritmo':<22}{'MAE%':>8}{'ingenuo':>10}{'ganho':>9}")
print("-" * 94)
comparacao = []
for setor, tabela in tabelas.items():
    melhor = tabela.iloc[0]
    ingenuo = float(tabela.set_index("algoritmo").loc[INGENUO, "mae_percentual"])
    ganho = (ingenuo - melhor.mae_percentual) / ingenuo * 100
    comparacao.append({"setor": setor, "melhor": melhor.algoritmo,
                       "mae_pct": melhor.mae_percentual, "ingenuo_pct": ingenuo,
                       "ganho_pct": round(ganho, 1)})
    print(f"{setor:<16}{melhor.algoritmo:<22}{melhor.mae_percentual:>8.2f}"
          f"{ingenuo:>10.2f}{ganho:>8.1f}%")

print("\nLeitura honesta deste quadro: o ganho sobre o ingenuo e MODESTO — da ordem de 1 a 15%.")
print("Numa serie de 15 minutos a inercia do consumo explica quase tudo, e e exatamente por isso")
print("que o ingenuo tinha de estar na comparacao. Sem ele, um R2 de 0,97 pareceria um triunfo.")

---
## 9. Resultados

### 9.1 Erro percentual por setor e por algoritmo

In [ ]:
algoritmos = [INGENUO, *candidatos()]
nomes_setores = list(tabelas)
largura = 0.8 / len(algoritmos)
posicoes = np.arange(len(nomes_setores))

figura, eixo = plt.subplots(figsize=(12, 4.6))
for i, algoritmo in enumerate(algoritmos):
    valores = [float(tabelas[s].set_index("algoritmo").loc[algoritmo, "mae_percentual"])
               for s in nomes_setores]
    barras = eixo.bar(posicoes + i * largura, valores, largura * 0.92, label=algoritmo,
                      color="0.6" if algoritmo == INGENUO else None,
                      hatch="//" if algoritmo == INGENUO else None)
    eixo.bar_label(barras, fmt="%.1f", fontsize=6, padding=1)

eixo.set_xticks(posicoes + largura * (len(algoritmos) - 1) / 2)
eixo.set_xticklabels(nomes_setores)
eixo.set_ylabel("MAE% (menor e melhor)")
eixo.set_title("Figura 4 — Erro percentual por setor e por algoritmo, no periodo de teste", fontsize=11)
eixo.legend(fontsize=8, ncols=5, loc="upper center", bbox_to_anchor=(0.5, -0.1))
figura.tight_layout()
plt.show()

print("A barra cinza hachurada e o ingenuo. Repare que nas Utilidades praticamente todos os")
print("algoritmos empatam com ele: e a serie mais plana, e ali nao ha padrao a aprender alem")
print("da propria inercia.")

### 9.2 Real × previsto e importância dos atributos

O algoritmo escolhido para os quatro setores é o **Gradient Boosting**: ele fica no primeiro ou
segundo lugar em todos, e o arquivo salvo tem **0,19 MB** contra **mais de 220 MB** do Random
Forest, com previsão individual de **0,22 ms** contra 12,5 ms. Para um sistema com requisito de
resposta abaixo de 500 ms e que carrega quatro modelos na subida, essa diferença decide.

In [ ]:
ESCOLHIDO = "Gradient Boosting"
finais = {}

figura, eixos = plt.subplots(2, 4, figsize=(16, 7))
for coluna, (setor, quadro) in enumerate(preparados.items()):
    X_treino, y_treino, X_teste, y_teste = separar(quadro)
    pipeline = candidatos()[ESCOLHIDO]
    pipeline.fit(X_treino, y_treino)
    previsto = np.maximum(0.0, pipeline.predict(X_teste))
    finais[setor] = pipeline

    datas = quadro[quadro.conjunto == "teste"].date
    fatia = slice(0, 96 * 3)  # 3 dias

    eixos[0, coluna].plot(datas.iloc[fatia], y_teste.iloc[fatia].values, lw=1.0, label="real")
    eixos[0, coluna].plot(datas.iloc[fatia], previsto[fatia], lw=1.0, ls="--", label="previsto")
    eixos[0, coluna].set_title(f"{setor} — 3 dias do teste", fontsize=9)
    eixos[0, coluna].tick_params(labelsize=6, axis="x", rotation=30)
    eixos[0, coluna].legend(fontsize=7)

    importancia = pd.Series(pipeline.named_steps["modelo"].feature_importances_, index=ATRIBUTOS)
    importancia.sort_values().plot.barh(ax=eixos[1, coluna], color="tab:blue")
    eixos[1, coluna].set_title(f"{setor} — importancia dos atributos", fontsize=9)
    eixos[1, coluna].tick_params(labelsize=7)
    eixos[1, coluna].set_xlim(0, 1)

figura.suptitle(f"Figura 5 — {ESCOLHIDO}: real x previsto e importancia dos atributos", fontsize=12)
figura.tight_layout()
plt.show()

In [ ]:
print("Importancia dos atributos no Gradient Boosting de cada setor\n" + "-" * 94)
importancias = pd.DataFrame({
    setor: pd.Series(pipeline.named_steps["modelo"].feature_importances_, index=ATRIBUTOS)
    for setor, pipeline in finais.items()
}).round(4)
display(importancias)

print("\nlag_1 sozinho responde por 94% a 98% da importancia nos quatro setores. Os atributos de")
print("calendario somam menos de 5%. Esse e o resultado mais importante do trabalho, e ele e")
print("desconfortavel: o modelo e, na pratica, uma correcao fina sobre 'repita o valor anterior'.")
print("E por isso que o MAE% fica tao perto do ingenuo.")

---
## 10. Da previsão ao alerta: o que o número vira no sistema

O modelo não é o produto final: ele alimenta um sistema de alertas. Duas regras usam a saída da
análise, e a própria análise mostrou que elas têm consequências muito diferentes.

### 10.1 O limite de consumo é o percentil 99 **do período de treino**

Um limite global não serviria: entre os clientes completos a escala varia mais de 8.000×. O limite
de cada setor é o **percentil 99 do seu próprio consumo no período de treino** — nunca no de teste,
para não escolher o limite olhando os dados em que o modelo será avaliado.

O percentil 99 foi escolhido pelo **volume de alertas que produz**: 1% de 672 períodos semanais dá
cerca de 7 alertas por semana por setor. O percentil 95 daria ~34, o que tornaria a tela de
resolver alertas inutilizável.

In [ ]:
PERCENTIL = 0.99
PERIODOS_POR_SEMANA = 7 * 96

print(f"{'setor':<16}{'media':>9}{'limite p99':>12}{'acima/ano':>11}{'%':>8}{'por semana':>12}")
print("-" * 70)
limites = {}
for setor, quadro in preparados.items():
    treino = quadro[quadro.conjunto == "treino"][ALVO]
    limite = float(treino.quantile(PERCENTIL))
    limites[setor] = limite
    serie = series[setor]
    acima = int((serie > limite).sum())
    print(f"{setor:<16}{serie.mean():>9.2f}{limite:>12.2f}{acima:>11}"
          f"{acima / len(serie) * 100:>7.2f}%{acima / len(serie) * PERIODOS_POR_SEMANA:>12.1f}")

print("\nOs quatro caem entre 5 e 8 alertas por semana. A regra e utilizavel nesse volume.")

### 10.2 O fator de potência mínimo de 0,92 — e por que ele é um **resultado**, não um detalhe

A ANEEL exige fator de potência de no mínimo **0,92**; abaixo disso a distribuidora cobra por
energia reativa excedente. Só o setor Produção tem esse dado medido.

O que a análise encontrou é o achado mais relevante do trabalho para a indústria: o fator de
potência defasado fica abaixo de 0,92 em **60,4% dos 35.040 períodos do ano**. E não é efeito só de
carga baixa — a célula abaixo mede por decil de consumo.

In [ ]:
quadro_fp = pd.DataFrame({
    "kwh": producao_bruto["Usage_kWh"].to_numpy(),
    "fp": (producao_bruto["Lagging_Current_Power_Factor"] / 100).to_numpy(),
})
abaixo = quadro_fp.fp < 0.92

print(f"Periodos no ano                          : {len(quadro_fp):,}")
print(f"Com fator de potencia abaixo de 0,92     : {int(abaixo.sum()):,}  ({abaixo.mean() * 100:.1f}%)")
print(f"Equivalente por semana                   : {abaixo.mean() * PERIODOS_POR_SEMANA:.0f} alertas")
print(f"Fator de potencia minimo da base         : {quadro_fp.fp.min():.4f}")
print(f"Exatamente 0,92 (nao gera alerta)        : {int((quadro_fp.fp == 0.92).sum())} periodos")

quadro_fp["decil"] = pd.qcut(quadro_fp.kwh, 10, labels=False, duplicates="drop") + 1
por_decil = quadro_fp.groupby("decil").agg(
    consumo_medio_kwh=("kwh", "mean"),
    fp_medio=("fp", "mean"),
    pct_abaixo_092=("fp", lambda s: (s < 0.92).mean() * 100),
).round(2)
print("\nPercentual de periodos abaixo de 0,92, por decil de consumo:")
display(por_decil)

print(f"\nNo decil de MAIOR consumo ainda sao {por_decil.pct_abaixo_092.iloc[-1]:.0f}% dos periodos.")
print("Ou seja: nao e artefato de carga baixa. Exigir um consumo minimo para disparar o alerta")
print("levaria o volume a ~140 por semana — cobraria o preco de mudar a regra sem resolver o")
print("problema. A regra fica como esta, e os 60,4% sao apresentados como RESULTADO: uma planta")
print("que, sob a regulacao brasileira, pagaria multa por energia reativa em boa parte do ano.")

In [ ]:
figura, (serie_fp, dispersao) = plt.subplots(1, 2, figsize=(13, 4.4))

mes = producao_bruto.sort_values("date").set_index("date").loc["2018-03"]
serie_fp.plot(mes.index, mes["Lagging_Current_Power_Factor"] / 100, lw=0.6)
serie_fp.axhline(0.92, color="tab:red", lw=1.2, ls="--", label="minimo ANEEL = 0,92")
serie_fp.set_title("Fator de potencia defasado — marco/2018", fontsize=10)
serie_fp.set_ylabel("fator de potencia")
serie_fp.tick_params(axis="x", rotation=30, labelsize=7)
serie_fp.legend(fontsize=8)

dispersao.scatter(quadro_fp.kwh, quadro_fp.fp, s=1, alpha=0.15)
dispersao.axhline(0.92, color="tab:red", lw=1.2, ls="--")
dispersao.set_xlabel("consumo no periodo (kWh)")
dispersao.set_ylabel("fator de potencia")
dispersao.set_title("Fator de potencia x consumo (35.040 periodos)", fontsize=10)

figura.suptitle("Figura 6 — O fator de potencia da siderurgica e o minimo regulatorio", fontsize=11)
figura.tight_layout()
plt.show()

---
## 11. Os quatro modelos salvos

Cada `.pkl` é um **pipeline completo** (padronização + modelo), salvo com `joblib`. Ele recebe as
nove colunas cruas e devolve kWh — nenhum pré-processamento fica de fora do arquivo, o que evita
que o sistema e o notebook divirjam.

In [ ]:
PASTA_MODELOS = Path("modelos")
PASTA_MODELOS.mkdir(exist_ok=True)

registro = {}
print(f"{'setor':<16}{'arquivo':<40}{'MB':>7}{'ms/prev':>10}{'confere':>9}")
print("-" * 84)
for setor, pipeline in finais.items():
    caminho = PASTA_MODELOS / f"modelo_consumo_{setor}.pkl"
    joblib.dump(pipeline, caminho)

    # Prova de que o arquivo salvo preve o mesmo que o objeto em memoria.
    recarregado = joblib.load(caminho)
    X_teste = separar(preparados[setor])[2]
    confere = bool(np.allclose(pipeline.predict(X_teste), recarregado.predict(X_teste)))

    uma_linha = X_teste.iloc[[0]]
    recarregado.predict(uma_linha)
    inicio = time.time()
    for _ in range(50):
        recarregado.predict(uma_linha)
    ms = (time.time() - inicio) / 50 * 1000

    mb = caminho.stat().st_size / 1024**2
    tabela = tabelas[setor].set_index("algoritmo").loc[ESCOLHIDO]
    registro[setor] = {
        "algoritmo": ESCOLHIDO, "arquivo": caminho.name, "mb": round(mb, 3),
        "ms_por_previsao": round(ms, 2), "confere_ao_recarregar": confere,
        "mae_kwh": float(tabela.mae_kwh), "mae_percentual": float(tabela.mae_percentual),
        "rmse": float(tabela.rmse), "r2": float(tabela.r2),
        "limite_consumo_kwh": round(limites[setor], 2),
    }
    print(f"{setor:<16}{caminho.name:<40}{mb:>7.3f}{ms:>10.2f}{str(confere):>9}")

resumo_final = {
    "algoritmo": ESCOLHIDO,
    "versao_scikit_learn": sklearn.__version__,
    "semente": SEMENTE,
    "atributos": ATRIBUTOS,
    "alvo": ALVO,
    "separacao": "10 primeiros meses para treino, 2 ultimos para teste, nunca embaralhado",
    "setores": registro,
}
Path("modelos_finais_notebook.json").write_text(json.dumps(resumo_final, indent=2, ensure_ascii=False))

print("\nResumo final:")
display(pd.DataFrame(registro).T[["mae_kwh", "mae_percentual", "rmse", "r2",
                                  "limite_consumo_kwh", "ms_por_previsao"]])
print("\nSalvo em modelos/ e em modelos_finais_notebook.json")

---
## 12. Síntese dos resultados

| # | Resultado | Número |
|---|---|---|
| 1 | Instâncias analisadas | 35.040 períodos × 4 setores = **140.160** registros |
| 2 | Arquivo processado em fluxo | **678,1 MB**, 140.256 × 371, lido com pico de **256 MB** de RAM contra 1.037 MB de uma vez |
| 3 | Clientes descartados por incompletude | 30 de 370 (o `MT_223` com 100% de zeros) |
| 4 | Dispersão de escala entre clientes completos | **8.238×** (5,9 kW a 48.306,7 kW) |
| 4b | Autocorrelação de defasagem 1 | **0,912 a 0,980** nos quatro setores |
| 5 | Algoritmo escolhido para os 4 setores | **Gradient Boosting** |
| 6 | Erro percentual (MAE%) | **4,02%** (Manutenção) a **20,23%** (Produção) |
| 7 | R² | 0,875 a 0,971 |
| 8 | Ganho sobre a linha de base ingênua | apenas **1% a 15%** |
| 9 | Importância do `lag_1` | **94% a 98%** |
| 10 | Fator de potência abaixo de 0,92 na siderúrgica | **60,4%** dos períodos do ano |
| 11 | Tamanho de cada modelo salvo | **0,19 MB**, 0,22 ms por previsão |

### As três conclusões que importam

**1. Um modelo por setor se justifica — mas pela forma, não pela escala.** O painel normalizado da
Figura 1 mostra que as quatro curvas têm formatos diferentes, e não apenas alturas diferentes. Um
modelo único treinado nos quatro teria de aprender simultaneamente o degrau comercial do
Administrativo e a planície das Utilidades.

**2. O ganho do aprendizado de máquina sobre a inércia da série é modesto, e dizer isso é parte do
resultado.** Com `lag_1` respondendo por até 98% da importância e o ganho sobre o modelo ingênuo
ficando entre 1% e 15%, o que se construiu foi uma correção fina sobre "o próximo período parece
com o anterior". Num horizonte de 15 minutos isso era esperado — e é justamente por isso que a
linha de base ingênua tinha de estar na comparação. Sem ela, um R² de 0,97 pareceria um triunfo do
método, quando boa parte dele vem da estrutura do problema.

**3. O achado de maior valor prático não veio do modelo, veio da análise exploratória.** O fator de
potência abaixo de 0,92 em 60,4% dos períodos, persistindo em 53% até no decil de maior consumo, é
um problema de conformidade regulatória que custa dinheiro — e foi encontrado olhando a
distribuição dos dados, não treinando um regressor.

### Limitações

- Horizonte de **um único período** (15 minutos) à frente. Horizontes maiores exigiriam previsão
  recursiva e degradariam rápido, dada a dependência de `lag_1`.
- Os hiperparâmetros são os **padrões do scikit-learn**. Não houve busca em grade, nem validação
  cruzada temporal — o critério de escolha incluiu tamanho e latência, não só erro.
- **Um ano por setor**, sem variação interanual: não há como separar tendência de sazonalidade.
- Três dos quatro setores são clientes anônimos de Portugal **representando** setores industriais,
  não medições de uma mesma planta.

---

## Checklist da entrega (Ulife — "ENTREGA DA A3 - ARQUIVOS")

- [ ] **Artigo** em PDF, 10 a 12 páginas, template SBC do Overleaf
- [ ] **Data set**: `Steel_industry_data.csv`, `administrativo.csv`, `utilidades.csv`, `manutencao.csv`
- [ ] **Script**: este notebook (`.ipynb`)
- [ ] **Apresentação**: 10 a 20 minutos, com tema, link das bases, nº de atributos, instâncias,
      valores e a informação de que **não há atributos multivalorados**